In [1]:
import numpy as np
from scipy.integrate import solve_ivp
import sys
sys.path.insert(0, '../src')
from cr3bp import cr3bp_rhs, jacobi_constant, find_lagrange_points, MU_EARTH_MOON

mu = MU_EARTH_MOON
lps = find_lagrange_points(mu)

# Your converged orbit
x0 = 0.8359151258
vy0 = 0.0084348855
T = 2.6917900928
state0 = [x0, 0.0, 0.0, vy0]

print("=" * 55)
print("SELF-CONSISTENCY VALIDATION: L1 Lyapunov orbit")
print("=" * 55)

# Integrate one full period at high resolution
sol = solve_ivp(cr3bp_rhs, (0, T), state0, args=(mu,),
                method='DOP853', rtol=1e-12, atol=1e-12,
                t_eval=np.linspace(0, T, 2000))

# --- Check 1: perpendicular crossing at start (vx0 should be 0 by construction) ---
print(f"\n[1] Perpendicular start: vx0 = {state0[2]:.2e}  (should be 0)")

# Find the half-period crossing (y=0 again) and check vx there
# The orbit crosses y=0 at t=0 (start) and t=T/2 (half period)
yvals = sol.y[1]
# Find sign change in y after the start
sign_changes = np.where(np.diff(np.sign(yvals)))[0]
if len(sign_changes) > 0:
    # interpolate vx at the first crossing after start
    idx = sign_changes[0]
    # linear interpolation for vx at y=0
    frac = -yvals[idx] / (yvals[idx+1] - yvals[idx])
    vx_at_cross = sol.y[2][idx] + frac * (sol.y[2][idx+1] - sol.y[2][idx])
    print(f"    Perpendicular half-period crossing: vx = {vx_at_cross:.2e}  (should be ~0)")

# --- Check 2: full-period closure ---
final_state = sol.y[:, -1]
closure_error = np.linalg.norm(final_state - np.array(state0))
print(f"\n[2] Full-period closure error: {closure_error:.2e}  (should be < 1e-9)")

# --- Check 3: Jacobi conservation around the orbit ---
C_vals = np.array([jacobi_constant(sol.y[:, i], mu) for i in range(sol.y.shape[1])])
C_variation = C_vals.max() - C_vals.min()
print(f"\n[3] Jacobi constant around orbit:")
print(f"    mean C = {C_vals.mean():.10f}")
print(f"    max variation = {C_variation:.2e}  (should be < 1e-9)")

# --- Check 4: period vs linearized prediction (small-amplitude) ---
omega = 2.334386
T_linear = 2 * np.pi / omega
print(f"\n[4] Period vs linear prediction:")
print(f"    nonlinear T = {T:.6f}")
print(f"    linear T    = {T_linear:.6f}")
print(f"    difference  = {abs(T - T_linear):.2e}  (small for small orbit)")

print("\n" + "=" * 55)

SELF-CONSISTENCY VALIDATION: L1 Lyapunov orbit

[1] Perpendicular start: vx0 = 0.00e+00  (should be 0)
    Perpendicular half-period crossing: vx = 0.00e+00  (should be ~0)

[2] Full-period closure error: 1.77e-07  (should be < 1e-9)

[3] Jacobi constant around orbit:
    mean C = 3.1882812229
    max variation = 1.31e-13  (should be < 1e-9)

[4] Period vs linear prediction:
    nonlinear T = 2.691790
    linear T    = 2.691579
    difference  = 2.11e-04  (small for small orbit)

